# MLOps Pipeline: Loan Default Retraining Pipeline

This notebook coordinates the complete machine learning lifecycle for the Loan Default Prediction model. It is designed to run entirely on **Google Colab (12GB RAM)**, leveraging PostgreSQL database (Supabase) and experiment tracking registry (DagsHub/MLflow).

### Pipeline Steps:
1. **Environment Setup**: Install packages and configure environment variables.
2. **Data Ingestion**: Download the Home Credit dataset from Kaggle and load it into Supabase PostgreSQL.
3. **Data Validation**: Run Great Expectations to assert schema and feature sanity.
4. **Model Training**: Train XGBoost Classifier and log parameters, metrics, and models to MLflow.
5. **Data Drift Detection**: Run Evidently AI to generate drift reports.

## Step 0: Diagnostics (Verify Colab VM & RAM)
Run the cell below to print the server specifications and confirm if you are running on Google's cloud server or locally.

In [1]:
import os
import socket
import platform
import psutil

print("=== SYSTEM DIAGNOSTIC ===")
hostname = socket.gethostname()
os_platform = platform.platform()
cpu_count = os.cpu_count()
total_ram = psutil.virtual_memory().total / (1024**3)

print(f"Hostname: {hostname}")
print(f"OS Platform: {os_platform}")
print(f"CPU Count: {cpu_count}")
print(f"Total RAM Available: {total_ram:.2f} GB")

if "colab" in hostname.lower() or os_platform.lower().startswith("linux"):
    print("\nRESULT: Connected to Google Colab cloud server!")
else:
    print("\nRESULT: Running on local machine. Please select the Google Colab kernel.")

=== SYSTEM DIAGNOSTIC ===
Hostname: 6acf11bdac02
OS Platform: Linux-6.6.122+-x86_64-with-glibc2.35
CPU Count: 2
Total RAM Available: 12.67 GB

RESULT: Connected to Google Colab cloud server!


## Step 1: Environment Setup
First, we install all the required MLOps dependencies.

In [3]:
# Install dependencies in Google Colab
!pip install -q xgboost scikit-learn mlflow psycopg2-binary great-expectations evidently pandas python-dotenv gdown

### Step 1.2: Define Credentials
Run this cell and enter your Supabase and DagsHub credentials.

In [ ]:
import os

# --- SUPABASE DATABASE CREDENTIALS ---
os.environ["DB_HOST"] = ""
os.environ["DB_PORT"] = ""
os.environ["DB_USER"] = ""
os.environ["DB_NAME"] = "postgres"
os.environ["DB_PASSWORD"] = "" # <--- Paste your Supabase password here

# --- DAGSHUB / MLFLOW EXPERIMENT CREDENTIALS ---
os.environ["MLFLOW_TRACKING_URI"] = ""       # <--- Paste DagsHub MLflow URI here
os.environ["MLFLOW_TRACKING_USERNAME"] = ""   # <--- Paste DagsHub username here
os.environ["MLFLOW_TRACKING_PASSWORD"] = ""      # <--- Paste DagsHub token or password here

os.environ["DRIFT_THRESHOLD"] = "0.05"

print("Credentials set successfully!")


Credentials set successfully!


## Step 2: Download & Ingest Kaggle Dataset
Paste your Kaggle API token (starting with `KGAT_`) when prompted. We will use it to download and extract the dataset onto the Colab server directly.

In [ ]:
import os
from getpass import getpass

# Prompt for the Kaggle API token
kaggle_token = getpass("Enter Kaggle API Token (starts with KGAT_): ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token

# Download and extract competition dataset
!kaggle competitions download -c home-credit-default-risk
!mkdir -p data
!unzip -o home-credit-default-risk.zip -d data/

# Verify files
!ls -lh data/

In [3]:
import sys

# Delete the cached version of the ingestion modules from Python's memory
for module_name in ['src.ingestion', 'src.database', 'src.config', 'src']:
    if module_name in sys.modules:
        del sys.modules[module_name]

# Now import the fresh code from the disk
from src.ingestion import ingest_data


### Step 2.2: Stream Data to Supabase
Now we run the ingestion pipeline. On Colab's fast internet connection, we will stream the dataset chunks directly into Supabase.

In [4]:
import sys
import os

# Append active directory to sys.path so Colab can import 'src'
sys.path.append(os.getcwd())

from src.ingestion import ingest_data

# Let's load 50,000 rows to Supabase
ingest_data(limit_records=50000)

In [5]:
!pip install -q great-expectations==0.18.15


In [6]:
import sys

# Force Python to forget the old cached configurations
for module in list(sys.modules.keys()):
    if module.startswith("src.") or module == "src":
        sys.modules.pop(module, None)

print("Cache cleared successfully! ")


Cache cleared successfully! 


## Step 3: Run Data Validation
Before training, we pull the dataset from Supabase and run Great Expectations to ensure the columns and distributions are valid.

In [7]:
import sys
import os
sys.path.append(os.getcwd())

from src.database import fetch_data_from_db
from src.validation import validate_data

# Fetch data
df = fetch_data_from_db("loans")

# Run Great Expectations validation
is_valid = validate_data(df)
print(f"Data Validation Passed: {is_valid}")

  df = pd.read_sql_query(query, conn)



Data Validation Passed: True


## Step 4: Model Training (XGBoost & MLflow)
This runs the training script which preprocessing the data, trains the XGBoost Classifier, logs all hyperparameters and metrics to DagsHub, and registers the model in the MLflow Model Registry.

In [8]:
import sys
import os
sys.path.append(os.getcwd())

from src.train import train_model

run_id = train_model()
print(f"Model retrained successfully. MLflow Run ID: {run_id}")

  return datetime.utcnow().replace(tzinfo=utc)

  df = pd.read_sql_query(query, conn)

  return datetime.utcnow().replace(tzinfo=utc)

2026/06/25 09:27:28 INFO mlflow.tracking.fluent: Experiment with name 'loan_default_prediction' does not exist. Creating a new experiment.
  return datetime.utcnow().replace(tzinfo=utc)

Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

  return datetime.utcnow().replace(tzinfo=utc)

2026/06/25 09:27:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
  return datetime.utcnow().replace(tzinfo=utc)

Successfully registered model 'loan_default_xgboost'.
2026/06/25 09:27:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: loan_default_xgboost, version 1
Created version '1' of model 'loan_default_xgboost'.


🏃 View run bustling-squid-613 at: https://dagshub.com/Abk700007/TCS_Xcelerate.mlflow/#/experiments/0/runs/3f2f9713679744b4bb9837df1e6ba4c6
🧪 View experiment at: https://dagshub.com/Abk700007/TCS_Xcelerate.mlflow/#/experiments/0
Model retrained successfully. MLflow Run ID: 3f2f9713679744b4bb9837df1e6ba4c6


In [3]:
!pip install -U evidently


In [19]:
import sys
for module in list(sys.modules.keys()):
    if module.startswith("src.") or module == "src":
        sys.modules.pop(module, None)
print("Cache cleared! ")


Cache cleared! 


## Step 5: Check Data Drift (Evidently AI)
We compare our training dataset (reference) with incoming application data (current) to see if borrower demographic behavior has shifted, which would trigger automatic retraining.

In [20]:
import sys
import os
sys.path.append(os.getcwd())

from src.drift import check_for_drift

drift_detected = check_for_drift()
print(f"Data Drift Detected: {drift_detected}")

/content/src/database.py:99: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
2026-06-25 14:00:29,813 - INFO - Successfully fetched 50000 records from table 'loans'.
2026-06-25 14:00:29,829 - INFO - Comparing Reference (40000 rows) and Current (10000 rows) datasets...
2026-06-25 14:00:32,122 - INFO - Evidently drift dashboard saved to reports/data_drift_report.html
2026-06-25 14:00:32,130 - INFO - Dataset Drift Detected: False (Drifted Feature Share: 0.00%)


Data Drift Detected: False


### View the Evidently AI Dashboard
Evidently has generated a detailed visual dashboard under `reports/data_drift_report.html`. You can download this file from the left sidebar file menu in Colab and double click it to view it in your browser.